# Homework 7 — Map projections

**Task.** Adapt the small multiple chart to show unemployment rates and change
the projection types. Observe the changes and appreciate the distances among
regions. Choose a favourite projection and explain the decision. Add a main
title. Export a single HTML file.

The data is county-level US unemployment (3,218 counties, FIPS-coded), joined
to the `us-10m` TopoJSON by county id.

In [1]:
"""Homework 7 - US unemployment under different map projections."""
import numpy as np, pandas as pd, altair as alt
from vega_datasets import data as vega_data

counties = alt.topo_feature(vega_data.us_10m.url, 'counties')
states   = alt.topo_feature(vega_data.us_10m.url, 'states')
unemployment = vega_data.unemployment.url

# Vega projection name -> the PROJ definition of the same projection, used for
# the distortion measurements further down.
PROJECTIONS = {
    'albersUsa':          None,                      # composite, no single PROJ equivalent
    'albers':             '+proj=aea +lat_1=29.5 +lat_2=45.5 +lat_0=37.5 +lon_0=-96',
    'mercator':           '+proj=merc',
    'equalEarth':         '+proj=eqearth +lon_0=-96',
    'azimuthalEqualArea': '+proj=laea +lat_0=37.5 +lon_0=-96',
    'orthographic':       '+proj=ortho +lat_0=37.5 +lon_0=-96',
}

## Why each panel has to be its own chart

A projection is a property of a **chart**, not an encoding channel. `facet()`
and `repeat()` iterate over data fields, so neither can iterate over
projections — the small multiple has to be built as six separate charts,
concatenated.

Two further details that only show up once you render it:

**Auto-fitting.** Vega-Lite fits a projection to the extent of its data. The
county file includes Alaska, Hawaii and Puerto Rico, so a non-composite
projection has to span from the Aleutians to the Caribbean, and the mainland
shrinks to an unreadable smudge. The grid below is therefore restricted to the
contiguous states, where the six projections are directly comparable. Alaska
gets its own figure.

**Rotation.** Azimuthal projections default to centring on (0°, 0°) — off the
west coast of Africa — which leaves the US on the rim at a steep oblique angle.
Conic and pseudocylindrical projections carry a sensible centre in their
parameters, but the azimuthal ones need an explicit rotation of (−lon, −lat) of
the point you want in the middle.

In [2]:
# d3 rotates the globe before projecting. Conic and pseudocylindrical
# projections are already centred near the US by their PROJ parameters, but the
# azimuthal ones default to (0, 0) - off the west coast of Africa - which drops
# the US on the rim at a steep oblique angle. Rotating by (-lon, -lat) of the
# US centre brings it to the middle of the projection.
ROTATE = {
    'albersUsa':          None,
    'albers':             None,
    'mercator':           None,
    'equalEarth':         [96, 0, 0],
    'azimuthalEqualArea': [96, -37.5, 0],
    'orthographic':       [96, -37.5, 0],
}

In [3]:
# --- the small multiple ------------------------------------------------------
# A projection is a property of a chart, not an encoding channel, so facet() and
# repeat() cannot iterate over projections: each panel has to be its own chart.
#
# Vega-Lite auto-fits a projection to the extent of its data. The county file
# includes Alaska, Hawaii and Puerto Rico, so a non-composite projection has to
# span from the Aleutians to the Caribbean and the mainland shrinks to a smudge.
# The grid below therefore shows the contiguous states, where the six
# projections are directly comparable; the Alaska question gets its own figure.
CONTIGUOUS = ('floor(datum.id / 1000) != 2 && floor(datum.id / 1000) != 15 '
              '&& floor(datum.id / 1000) < 57')

def county_map(projection, width=300, height=195, contiguous=True, title=None):
    base = alt.Chart(counties).mark_geoshape(stroke=None).encode(
        alt.Color('rate:Q',
                  scale=alt.Scale(scheme='viridis', domain=[0, 0.18], clamp=True),
                  legend=alt.Legend(title='Unemployment rate', format='.0%',
                                    orient='bottom', direction='horizontal',
                                    gradientLength=240)),
        tooltip=[alt.Tooltip('id:N', title='County FIPS'),
                 alt.Tooltip('rate:Q', title='Rate', format='.1%')],
    ).transform_lookup(
        lookup='id',
        from_=alt.LookupData(unemployment, key='id', fields=['rate']),
    )
    outline = alt.Chart(states).mark_geoshape(
        fill=None, stroke='white', strokeWidth=0.4)
    if contiguous:
        base = base.transform_filter(CONTIGUOUS)
        outline = outline.transform_filter(
            'datum.id != 2 && datum.id != 15 && datum.id < 57')
    proj_args = {'type': projection}
    if ROTATE.get(projection):
        proj_args['rotate'] = ROTATE[projection]
    return alt.layer(base, outline).project(
        **proj_args
    ).properties(width=width, height=height, title=title or projection)

panels = [county_map(p) for p in PROJECTIONS]
grid = alt.vconcat(
    alt.hconcat(*panels[0:3], spacing=12),
    alt.hconcat(*panels[3:6], spacing=12),
    spacing=12,
).resolve_scale(color='shared')

chart = grid.properties(
    title=alt.TitleParams(
        text='US county unemployment under six map projections',
        subtitle=['Contiguous states only, identical data and identical colour '
                  'scale — only the projection changes.',
                  'Equal-area projections (albers, equalEarth, '
                  'azimuthalEqualArea) keep every county\'s share of the page '
                  'proportional to its real size; mercator and orthographic do not.'],
        anchor='start', fontSize=18, subtitleColor='gray', subtitleFontSize=12),
).configure_view(stroke=None)

chart

alt.VConcatChart(...)

## What albersUsa hides

`albersUsa` is not one projection — it is three, glued together, with Alaska
scaled down and Alaska and Hawaii relocated beneath the southwest. It is the
conventional choice for US maps and it is very readable, but it answers "how
big is Alaska" and "where is Alaska" incorrectly by construction. The
assignment asks us to appreciate the distances among regions, so here is the
same data under a single true projection for comparison.

In [4]:
# --- what albersUsa hides ----------------------------------------------------
# The assignment asks us to "appreciate the distances among regions". albersUsa
# is a composite: it cuts Alaska and Hawaii out, shrinks Alaska and parks both
# under the southwest. Any single true projection has to show where they are.
alaska = alt.hconcat(
    county_map('albersUsa', 330, 210, contiguous=False,
               title='albersUsa — composite, Alaska moved and resized'),
    county_map('equalEarth', 330, 210, contiguous=False,
               title='equalEarth — one true projection, real positions'),
    spacing=16,
).resolve_scale(color='shared').properties(
    title=alt.TitleParams(
        text='The same data, with Alaska, Hawaii and Puerto Rico included',
        subtitle=['albersUsa is three projections glued together. On the left '
                  'Alaska is shrunk and parked below California; on the right '
                  'it sits where it really is, far to the northwest,',
                  'and at its true size — about a fifth the area of all the '
                  'contiguous states combined.'],
        anchor='start', fontSize=15, subtitleColor='gray', subtitleFontSize=11),
).configure_view(stroke=None)

alaska

alt.HConcatChart(...)

## Measuring the distortion

Rather than assert which projections preserve area, measure it. For each
projection the local area scale factor is computed on a grid over the
contiguous US by finite differences: transform a point and two neighbours a
small step away in longitude and latitude, take the Jacobian determinant of the
mapping, and divide by the true area element `cos(lat)`.

For an equal-area projection that factor is constant everywhere. The **spread**
— largest over smallest — is how unevenly a projection inflates one region
relative to another.

In [5]:
# --- how much does each projection distort area? -----------------------------
def area_distortion(proj_string, lon_range=(-125, -67), lat_range=(25, 49), n=28):
    """Local area scale factor on a grid, by finite differences.

    For an equal-area projection the factor is constant; the spread between its
    largest and smallest value is how unevenly a projection inflates regions."""
    from pyproj import Transformer
    tr = Transformer.from_crs('EPSG:4326', proj_string, always_xy=True)
    lons = np.linspace(*lon_range, n)
    lats = np.linspace(*lat_range, n)
    d = 0.01
    factors = []
    for lat in lats:
        for lon in lons:
            x0, y0 = tr.transform(lon, lat)
            x1, y1 = tr.transform(lon + d, lat)      # d/dlon
            x2, y2 = tr.transform(lon, lat + d)      # d/dlat
            if not all(np.isfinite([x0, y0, x1, y1, x2, y2])):
                continue
            # Jacobian determinant, divided by the true area element cos(lat)
            jac = abs((x1 - x0) * (y2 - y0) - (x2 - x0) * (y1 - y0)) / d**2
            factors.append(jac / np.cos(np.radians(lat)))
    f = np.array(factors)
    return f / f.mean()

In [6]:
import pandas as pd, numpy as np
rows = []
for name, proj in PROJECTIONS.items():
    if proj is None:
        rows.append({'projection': name, 'min': np.nan, 'max': np.nan,
                     'spread': np.nan, 'note': 'composite — measured as albers'})
        continue
    f = area_distortion(proj)
    rows.append({'projection': name, 'min': f.min(), 'max': f.max(),
                 'spread': f.max() / f.min(), 'note': ''})
pd.DataFrame(rows)

,projection,min,max,spread,note
0,albersUsa,NaN,NaN,NaN,composite — measured as albers
1,albers,0.997497,1.002708,1.005224,
2,mercator,0.742097,1.420012,1.913512,
3,equalEarth,0.997498,1.002708,1.005223,
4,azimuthalEqualArea,0.997479,1.002732,1.005267,
5,orthographic,0.917199,1.037695,1.131374,


## Result and preference

| projection | area spread | equal-area? |
|---|---|---|
| albers | 1.005 | yes |
| equalEarth | 1.005 | yes |
| azimuthalEqualArea | 1.005 | yes |
| orthographic | 1.131 | no |
| **mercator** | **1.914** | **no** |

The three equal-area projections come out at 1.005, which is the residual of
the finite-difference step and the ellipsoid — effectively 1.000, as theory
requires. Orthographic loses 13% toward the edges of the view.

**Mercator spreads by 1.91.** A county along the Canadian border occupies
roughly **91% more of the page per square kilometre** than a county on the Gulf
coast.

### Preference: Albers equal-area conic

For a choropleth the argument is not aesthetic. The visual weight of a region
is its **area on the page**, so if the projection inflates northern states, a
reader's impression of how much of the country is unemployed is inflated along
with them. Mercator is conformal — it preserves *angles*, which is what made it
useful for navigation, and it is the wrong trade for a map whose entire message
is carried by filled area.

Among the three equal-area candidates, Albers is the right one for this
particular map: its two standard parallels can be placed to bracket the
latitudes of the contiguous US, which keeps *shape* distortion low across
exactly the region being shown, while Equal Earth and azimuthal equal-area are
optimised for the whole globe. The conventional `albersUsa` is Albers plus the
Alaska and Hawaii insets — fine when the reader understands the insets, which is
why the figure above shows what it costs.

## Export

In [7]:
chart.save('solution.html')
alaska.save('albersusa_comparison.html')
print('saved solution.html, albersusa_comparison.html')

saved solution.html, albersusa_comparison.html
